In [0]:
# 07_analise_perguntas_negocio.py
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

Pergunta 1 — Os clusters da Loja 1 se replicam nas demais lojas?

In [0]:
# Q1 — Similaridade de composição dos clusters entre lojas (usando Loja 1 como referência)
clusters_pd = spark.table("silver.sop_forecast.clusters_sku").toPandas()

ref = clusters_pd[clusters_pd["loja_id"] == 1].set_index("item_id")["cluster_id"]

resultados_q1 = []
for loja in sorted(clusters_pd["loja_id"].unique()):
    if loja == 1:
        continue
    comp = clusters_pd[clusters_pd["loja_id"] == loja].set_index("item_id")["cluster_id"]
    comum = ref.index.intersection(comp.index)
    # concordância direta (mesmo rótulo) e invertida (rótulos trocados) — pega o melhor
    concord_direta   = (ref[comum] == comp[comum]).mean()
    concord_invertida = (ref[comum] == (1 - comp[comum])).mean()
    concordancia = max(concord_direta, concord_invertida)
    resultados_q1.append({"loja_id": loja, "concordancia_com_loja1": round(concordancia, 3)})

df_q1 = pd.DataFrame(resultados_q1)
print(df_q1.to_string(index=False))
print(f"\nConcordância média entre lojas: {df_q1['concordancia_com_loja1'].mean():.1%}")

Pergunta 2 — Distribuição de volume por loja e cluster

In [0]:
# Q2 — Volume total por loja e cluster
df_q2 = (
    spark.table("gold.sop_forecast.dim_item_loja")
    .groupBy("loja_id", "cluster_id")
    .agg(F.sum("volume_total").alias("volume_total_cluster"))
    .orderBy("loja_id", "cluster_id")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12,5))
for cluster_id, cor in zip([0,1], ["#2649B2", "#9D5CE6"]):
    sub = df_q2[df_q2["cluster_id"] == cluster_id]
    ax.bar(sub["loja_id"] + (cluster_id*0.35 - 0.175), sub["volume_total_cluster"],
           width=0.35, label=f"Cluster {cluster_id}", color=cor)
ax.set_xlabel("Loja"); ax.set_ylabel("Volume total (unidades)")
ax.set_title("Volume Total por Loja e Cluster (2013–2017)")
ax.set_xticks(range(1,11)); ax.legend()
plt.tight_layout(); plt.show()

display(spark.createDataFrame(df_q2))

Pergunta 3 — Existe correlação entre cluster e perfil da loja?

In [0]:
# Q3 — Limitação de dados: não há atributos de loja (região, porte, tipo) no dataset
print("O dataset não possui atributos descritivos da loja (ex: região, porte, tipo de "
      "estabelecimento) — apenas o identificador numérico (1 a 10). Portanto, não é "
      "possível testar correlação entre cluster e 'perfil de loja' com os dados disponíveis. "
      "Esta pergunta permanece em aberto e é registrada como limitação (ver Autoavaliação).")

Pergunta 4 — Itens com histórico problemático (esparso/zeros) por loja

In [0]:
# Q4 — % de combinações loja-item com muitos meses de venda zero
vendas_mensal = spark.table("gold.sop_forecast.fato_vendas_mensal").toPandas()

resumo_zeros = (
    vendas_mensal.assign(mes_zero=lambda d: d["vendas_mes"] == 0)
    .groupby(["loja_id", "item_id"])["mes_zero"].sum()
    .reset_index(name="meses_com_zero")
)
resumo_zeros["pct_meses_zero"] = resumo_zeros["meses_com_zero"] / 60 * 100

limite_esparso = 10  # % arbitrário de corte para "esparso"
itens_esparsos = resumo_zeros[resumo_zeros["pct_meses_zero"] > limite_esparso]

print(f"Combinações loja-item com >{limite_esparso}% de meses zerados: {len(itens_esparsos)} de {len(resumo_zeros)}")
print(f"Percentual do portfólio: {len(itens_esparsos)/len(resumo_zeros)*100:.2f}%")

Pergunta 5 — O baseline Naive + mix mantém MAPE competitivo em escala?

In [0]:
# Q5 — Já resondida no notebook 05, aqui consolidamos a visão executiva
forecast_pd = spark.table("gold.sop_forecast.fato_forecast_cluster").select(
    "loja_id","cluster_id","mape_naive","mape_hw"
).distinct().toPandas()

print(f"MAPE médio Naive Sazonal : {forecast_pd['mape_naive'].mean():.2f}%")
print(f"MAPE médio Holt-Winters  : {forecast_pd['mape_hw'].mean():.2f}%")
print(f"Ganho médio HW vs Naive  : {(forecast_pd['mape_naive'] - forecast_pd['mape_hw']).mean():+.2f}pp")
print(f"Faixa MAPE Naive          : {forecast_pd['mape_naive'].min():.2f}% – {forecast_pd['mape_naive'].max():.2f}%")

Pergunta 6 — Qualidade geral dos dados

In [0]:
# Q6 — Consolidação do notebook 06 (referência)
print("Ver notebook 06_qualidade_dados: 0 nulos, 0 duplicatas, 0 outliers acima de p99.9, "
      "0 gaps estruturais (913.000 registros esperados = 913.000 reais). "
      "Dataset validado como íntegro em todas as 10 lojas.")

In [0]:
# Q3 (revisão) — Meta-clustering de lojas por similaridade de estrutura sazonal
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Recarrega os clusters da camada Silver
clusters_pd = spark.table("silver.sop_forecast.clusters_sku").toPandas()

lojas = sorted(clusters_pd["loja_id"].unique())
matriz_ari = pd.DataFrame(index=lojas, columns=lojas, dtype=float)

for l1 in lojas:
    c1 = clusters_pd[clusters_pd["loja_id"] == l1].set_index("item_id")["cluster_id"]
    for l2 in lojas:
        c2 = clusters_pd[clusters_pd["loja_id"] == l2].set_index("item_id")["cluster_id"]
        comuns = c1.index.intersection(c2.index)
        matriz_ari.loc[l1, l2] = adjusted_rand_score(c1[comuns], c2[comuns])

print("Matriz ARI entre lojas:")
print(matriz_ari.round(2))

dist_matrix = 1 - matriz_ari.values
np.fill_diagonal(dist_matrix, 0)

linkage_matrix = linkage(dist_matrix[np.triu_indices(len(lojas), k=1)], method='average')

fig, ax = plt.subplots(figsize=(10,5))
dendrogram(linkage_matrix, labels=[f"Loja {l}" for l in lojas], ax=ax)
ax.set_title("Dendrograma — Similaridade de Estrutura Sazonal entre Lojas (via ARI)")
plt.tight_layout(); plt.show()